# 07 - Optional LoRA Style Intervention

This is a compact controlled-training notebook. It builds tiny plain-vs-OCN SFT datasets from the reward-pair table, trains a LoRA adapter on Qwen 0.5B, saves outputs to Drive, and logs training to W&B.

Treat this as a pilot intervention, not a paper-scale DPO experiment.

In [ ]:
from pathlib import Path
import os, sys, json, subprocess, textwrap

DEFAULT_REPO_URL = "https://github.com/ritwikraha/AutoRegressive-Bhasha.git"

def find_repo_root():
    try:
        import google.colab  # type: ignore  # noqa: F401
        from google.colab import drive  # type: ignore
        if not Path("/content/drive/MyDrive").exists():
            drive.mount("/content/drive")
    except Exception:
        pass

    candidates = [
        Path.cwd(),
        Path("/content/empty-negations"),
        Path("/content/AutoRegressive-Bhasha/empty-negations"),
        Path("/content/drive/MyDrive/ocn_empty_negations"),
        Path("/content/drive/MyDrive/AutoRegressive-Bhasha/empty-negations"),
    ]
    for candidate in candidates:
        if (candidate / "src/ocn").exists():
            return candidate
    repo_url = os.environ.get("OCN_REPO_URL", DEFAULT_REPO_URL)
    target = Path("/content/AutoRegressive-Bhasha")
    if not target.exists():
        subprocess.run(["git", "clone", "--depth", "1", repo_url, str(target)], check=True)

    cloned_candidates = [target / "empty-negations", target]
    for candidate in cloned_candidates:
        if (candidate / "src/ocn").exists():
            return candidate

    raise FileNotFoundError(
        f"Cloned {repo_url}, but could not find src/ocn. "
        "Set OCN_REPO_URL to a repository containing empty-negations/src/ocn."
    )

REPO_ROOT = find_repo_root()
sys.path.insert(0, str(REPO_ROOT / "src"))
print("Repo:", REPO_ROOT)

In [ ]:
import json
from pathlib import Path
import pandas as pd
import torch
import wandb
from datasets import Dataset, load_dataset
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from trl import SFTTrainer

from ocn.colab_utils import login_huggingface, login_wandb, make_colab_paths

paths = make_colab_paths()
config = json.loads((paths.project_root / "ocn_colab_config.json").read_text())
login_huggingface("HF_WRITE_ACCESS")
run = login_wandb(project="ocn-empty-negations", name=f"lora-sft-{config['run_id']}", config=config)

In [ ]:
pairs = load_dataset(config["hf_reward_pairs_repo"], split="train").to_pandas()

CONDITION = "ocn"  # use "plain" for the counter-condition
if CONDITION == "plain":
    train_df = pairs[pairs["variant_type"].eq("plain")].copy()
else:
    train_df = pairs[pairs["variant_type"].isin(["justified_ocn", "empty_ocn"])].copy()

train_df["text"] = "Prompt: Explain the topic clearly.\nAnswer: " + train_df["response"]
train_ds = Dataset.from_pandas(train_df[["text"]], preserve_index=False)
train_ds

In [ ]:
BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    device_map="auto",
    quantization_config=quant_config,
    trust_remote_code=True,
)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

output_dir = Path(config["drive_project_root"]) / f"adapters/qwen05_{CONDITION}_{config['run_id']}"
args = TrainingArguments(
    output_dir=str(output_dir),
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=3,
    logging_steps=1,
    save_strategy="epoch",
    report_to=["wandb"],
    bf16=True,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    dataset_text_field="text",
    peft_config=peft_config,
    args=args,
    max_seq_length=512,
)
trainer.train()
trainer.save_model(str(output_dir))
run.finish()
output_dir